### data_scrapping

- 주식 상위 30개 코인 상위 10개 데이터 다운로드

In [3]:
import yfinance as yf
import pandas as pd

sp500_tickers = [
    'NVDA',   # 엔비디아
    'AAPL',   # 애플
    'MSFT',   # 마이크로소프트
    'AMZN',   # 아마존
    'GOOGL',  # 알파벳 A
    'META',   # 메타 플랫폼스
    'BRK-B',  # 버크셔 해서웨이
    'LLY',    # 일라이 릴리
    'AVGO',   # 브로드컴
    'TSLA',   # 테슬라
    'WMT',    # 월마트
    'JPM',    # JP모건 체이스
    'UNH',    # 유나이티드헬스 그룹
    'XOM',    # 엑슨모빌
    'V',      # 비자
    'MA',     # 마스터카드
    'JNJ',    # 존슨앤드존슨
    'PG',     # 프록터 앤 갬블
    'ORCL',   # 오라클
    'HD',     # 홈디포
    'COST',   # 코스트코
    'ABBV',   # 애비비
    'BAC',    # 뱅크오브아메리카
    'NFLX',   # 넷플릭스
    'CVX',    # 쉐브론
    'AMD',    # AMD
    'KO',     # 코카콜라
    'CAT',    # 캐터필러
    'PLTR',   # 팔란티어
    'MU'      # 마이크론 테크놀로지
]

crypto_tickers = [
    'BTC-USD',   # 1. 비트코인 
    'ETH-USD',   # 2. 이더리움 
    'USDT-USD',  # 3. 테더 
    'BNB-USD',   # 4. 바이낸스 코인
    'XRP-USD',   # 5. 리플 
    'USDC-USD',  # 6. 유에스디 코인
    'SOL-USD',   # 7. 솔라나 
    'TRX-USD',   # 8. 트론
    'DOGE-USD',  # 9. 도지코인 
    'ADA-USD'    # 10. 카르다노
]

tickers = sp500_tickers + crypto_tickers

stock_data = yf.download(sp500_tickers, period='5y', interval='1d', auto_adjust=True)
crypto_data = yf.download(crypto_tickers, period='5y', interval='1d', auto_adjust=True)
data = pd.concat([stock_data, crypto_data], axis = 1)

[*********************100%***********************]  30 of 30 completed
[*********************100%***********************]  10 of 10 completed
/var/folders/z5/ctx2n2x97dg_4f3270h6bd7m0000gn/T/ipykernel_72627/1332215659.py:54: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  data = pd.concat([stock_data, crypto_data], axis = 1)


In [4]:
print(data.shape)
print(data.columns)
print(data.head())

(1827, 200)
MultiIndex([( 'Close',     'AAPL'),
            ( 'Close',     'ABBV'),
            ( 'Close',      'AMD'),
            ( 'Close',     'AMZN'),
            ( 'Close',     'AVGO'),
            ( 'Close',      'BAC'),
            ( 'Close',    'BRK-B'),
            ( 'Close',      'CAT'),
            ( 'Close',     'COST'),
            ( 'Close',      'CVX'),
            ...
            ('Volume',  'ADA-USD'),
            ('Volume',  'BNB-USD'),
            ('Volume',  'BTC-USD'),
            ('Volume', 'DOGE-USD'),
            ('Volume',  'ETH-USD'),
            ('Volume',  'SOL-USD'),
            ('Volume',  'TRX-USD'),
            ('Volume', 'USDC-USD'),
            ('Volume', 'USDT-USD'),
            ('Volume',  'XRP-USD')],
           names=['Price', 'Ticker'], length=200)
Price            Close                                               \
Ticker            AAPL       ABBV        AMD        AMZN       AVGO   
Date                                                       

- MultiIndex 중 사용할 'Close'(종가) column만 선택

In [5]:
stock_close = stock_data['Close']
crypto_close = crypto_data['Close']
data_close = data['Close']

- 결측치 확인 / stock과 crypto 데이터를 합치면서 broadcasting 때문에 결측치 발생 (stock 주말, 공휴일 등)

In [8]:
print("주식:",stock_close.isnull().sum().max())
print("코인:",crypto_close.isnull().sum().max())
print("전체:",data_close.isnull().sum().max())

주식: 0
코인: 0
전체: 571


- 종가 가격 기준 데이터 (주식/코인/전체)의 데이터 보존을 위해 .csv파일로 저장

In [10]:
stock_close.to_csv("Close_stock_data.csv")
crypto_close.to_csv("Close_crypto_data.csv")
data_close.to_csv("Close_data.csv")

- 결측치 처리(주말, 공휴일은 주식의 변동이 없었던 것으로 가정)

In [13]:
data_close_filled = data_close.ffill()
print("전체:",data_close_filled.isnull().sum().max())

전체: 0


In [14]:
data_close_filled.to_csv("Close_Filled_data.csv")

In [15]:
data_close_filled.head()

Ticker,AAPL,ABBV,AMD,AMZN,AVGO,BAC,BRK-B,CAT,COST,CVX,...,ADA-USD,BNB-USD,BTC-USD,DOGE-USD,ETH-USD,SOL-USD,TRX-USD,USDC-USD,USDT-USD,XRP-USD
Date,,,,,,,,,,,,,,,,,,,,,
2021-04-08,126.970055,87.623634,83.349998,164.964996,43.923389,35.159542,263.510010,210.199936,341.985138,83.656395,...,1.219436,418.038391,58323.953125,0.061464,2088.573730,27.033987,0.123070,1.000240,1.000167,1.052756
2021-04-09,129.541428,88.812851,82.760002,168.610001,43.888107,35.416374,266.010010,210.446167,343.869080,83.575188,...,1.203540,453.178467,58245.003906,0.061684,2072.108887,27.778275,0.115934,1.001493,1.001098,1.020837
2021-04-10,129.541428,88.812851,82.760002,168.610001,43.888107,35.416374,266.010010,210.446167,343.869080,83.575188,...,1.218758,472.560333,59793.234375,0.063845,2135.942139,26.841404,0.125983,1.000591,1.000469,1.374416
2021-04-11,129.541428,88.812851,82.760002,168.610001,43.888107,35.416374,266.010010,210.446167,343.869080,83.575188,...,1.266735,525.385559,60204.964844,0.074649,2157.656982,27.931890,0.122353,1.002870,1.002177,1.360530
2021-04-12,127.827202,89.399231,78.580002,168.969498,43.759632,35.478363,267.929993,210.610367,345.383972,82.649483,...,1.316127,598.722656,59893.453125,0.070767,2139.353271,28.512123,0.129048,1.000371,0.999937,1.467735


- 결측치 없는 가격 기준 데이터셋 생성 완료
- 해당 데이터를 `수익률` 기준 데이터셋으로 변경
$$
R_t = \frac{P_t}{P_{t-1}} - 1
$$ 

- `pct_change()` 내부적으로 data_close_filled / data_close_filled.shift(1) - 1 이렇게 동작 = 위 수식의 구현

In [17]:
returns = data_close_filled.pct_change()
stock_returns = stock_close.pct_change()
crypto_returns = crypto_close.pct_change()

- 전날 대비 수익률 기준으로 데이터를 바꿔주며, 전날 데이터가 없는 1행에 결측값이 생김
- isnull().sum().max() = 1 로 유일하게 1행에만 결측치가 발생한 것을 확인

In [25]:
print(stock_returns.iloc[:, :5].head(2))
print(f"결측치: {stock_returns.isnull().sum().max()}")
print()
print(crypto_returns.iloc[:, :5].head(2))
print(f"결측치: {crypto_returns.isnull().sum().max()}")
print()
print(returns.iloc[:,:5].head(2))
print(f"결측치: {returns.isnull().sum().max()}")

Ticker          AAPL      ABBV       AMD      AMZN      AVGO
Date                                                        
2021-04-08       NaN       NaN       NaN       NaN       NaN
2021-04-09  0.020252  0.013572 -0.007079  0.022096 -0.000803
결측치: 1

Ticker       ADA-USD   BNB-USD   BTC-USD  DOGE-USD   ETH-USD
Date                                                        
2021-04-08       NaN       NaN       NaN       NaN       NaN
2021-04-09 -0.013036  0.084059 -0.001354  0.003579 -0.007883
결측치: 1

Ticker          AAPL      ABBV       AMD      AMZN      AVGO
Date                                                        
2021-04-08       NaN       NaN       NaN       NaN       NaN
2021-04-09  0.020252  0.013572 -0.007079  0.022096 -0.000803
결측치: 1


- `.dropna()`를 통해 첫행 결측값을 제거
- 결측값이 제거된 수익률 기준 최종 데이터 점검 후 .csv파일로 저장하여 보존 

In [30]:
returns = returns.dropna()
stock_returns = stock_returns.dropna()
crypto_returns = crypto_returns.dropna()

In [29]:
print(stock_returns.shape)
print(stock_returns.iloc[:, :5].head(2))
print(f"결측치: {stock_returns.isnull().sum().max()}")
print()
print(crypto_returns.shape)
print(crypto_returns.iloc[:, :5].head(2))
print(f"결측치: {crypto_returns.isnull().sum().max()}")
print()
print(returns.shape)
print(returns.iloc[:,:5].head(2))
print(f"결측치: {returns.isnull().sum().max()}")

(1255, 30)
Ticker          AAPL      ABBV       AMD      AMZN      AVGO
Date                                                        
2021-04-09  0.020252  0.013572 -0.007079  0.022096 -0.000803
2021-04-12 -0.013233  0.006602 -0.050507  0.002132 -0.002927
결측치: 0

(1826, 10)
Ticker       ADA-USD   BNB-USD   BTC-USD  DOGE-USD   ETH-USD
Date                                                        
2021-04-09 -0.013036  0.084059 -0.001354  0.003579 -0.007883
2021-04-10  0.012644  0.042769  0.026581  0.035033  0.030806
결측치: 0

(1826, 40)
Ticker          AAPL      ABBV       AMD      AMZN      AVGO
Date                                                        
2021-04-09  0.020252  0.013572 -0.007079  0.022096 -0.000803
2021-04-10  0.000000  0.000000  0.000000  0.000000  0.000000
결측치: 0


In [31]:
stock_returns.to_csv("stock returns.csv")
crypto_returns.to_csv("crypto returns.csv")
returns.to_csv("returns.csv")